# Laya × Memory Fusion — All Full and Sliding Attention Layers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/codex/laya-all-attention/notebooks/Laya_MemoryFusion_Colab.ipynb)

Uses **`convaiinnovations/laya`** as the unchanged teacher. Attempts every ModernBERT full and sliding attention layer in encoder order.

- Full layers use bidirectional feature memory, sparse local mixing, and direct V.
- Sliding layers keep the original window radius in both mixing branches.
- Cached block fitting avoids a complete model forward on every optimizer step.
- Recurrent GDN2 is opt-in for full layers; disabled by default to avoid Python token loops.
- Best-probe restoration includes the memory activation flag.
- Each replacement must pass local fidelity **and cumulative student** decision gates. Failed layers retain original attention.
- `complete` means every attention layer was replaced; `partial` lists remaining original layers.

This is approximate distillation, not an exact mathematical substitution. Full conversion and faster inference are measured outcomes, not guarantees. Sparse softmax remains inside the replacement's local branch.


## 1. Setup
A T4/L4/A100 runtime is recommended. The setup pulls the latest TinyCeNN-LM and Laya source, then performs a syntax preflight before training.


In [ ]:
import os, sys, subprocess, pathlib, importlib, compileall
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if pathlib.Path("/content").exists():
    WORK = pathlib.Path("/content")
elif pathlib.Path("/kaggle/working").exists():
    WORK = pathlib.Path("/kaggle/working")
else:
    WORK = pathlib.Path.cwd()

REVISION = "codex/laya-all-attention"
REPO = WORK / "TinyCeNN-LM-MemoryFusion"
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "-q", "--branch", REVISION, "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REVISION])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", REVISION])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REVISION])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/NandhaKishorM/laya.git",
    "datasets", "pandas", "pyarrow", "safetensors"
])

SRC = str(REPO / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
# Reload this package after updates in a previously used runtime.
for name in list(sys.modules):
    if name == "tinycenn_lm" or name.startswith("tinycenn_lm."):
        del sys.modules[name]
importlib.invalidate_caches()

LAB_SRC = REPO / "src" / "tinycenn_lm" / "laya_lab"
assert compileall.compile_dir(str(LAB_SRC), quiet=1), "Python syntax preflight failed in tinycenn_lm/laya_lab"

import torch, json, pandas as pd
from tinycenn_lm.laya_lab.memory_fusion_v3 import (
    LayaMemoryFusionV3Config,
    run_memory_fusion_v3,
)

print("repo:", REPO)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2. All-attention configuration

`target_all_attention=True` discovers every full and sliding layer. For a quick single-layer test, set it to `False` and use `candidate_layer=12`; an explicit subset can use `target_layers=(0, 1, 2)` with all-layer mode off.

The router/gain learning rate starts at 0.01 after warm-up. Feature maps use 0.001 and copied output projections use 0.00005. Plateau cuts and best-probe restoration control regression. These are starting settings; this revision has not been trained on a T4 yet. Strict quality thresholds are unchanged.


In [ ]:
MODEL_ID = "convaiinnovations/laya"

cfg = LayaMemoryFusionV3Config(
    model_id=MODEL_ID,
    seed=2026,
    output_dir=str(WORK / "laya_tinycenn"),

    target_all_attention=True,
    target_layers=None,
    candidate_layer=12,       # used only when all-layer mode is off
    enable_recurrent_memory=False,  # opt-in, full layers only

    feature_dim=64,
    memory_rank=64,
    dilations=(1, 2, 4, 8, 16, 32, 64),

    train_cases=480,
    train_max_len=512,
    batch_size=4,
    train_cache_batches=48,   # 192 cached training examples
    probe_cache_batches=8,    # fixed 32-example probe
    cache_on_device=True,      # CPU fallback when cache exceeds GPU headroom

    # Turbo schedule: fewer steps, more useful movement per step.
    functional_steps=320,
    max_rounds=3,
    check_every=20,
    min_steps_before_check=40,
    fast_learning_rate=1.0e-2,   # fusion/router/gains
    core_learning_rate=1.0e-3,   # Hedgehog + GDN2 feature maps
    output_learning_rate=5.0e-5, # copied Laya Wo
    warmup_steps=10,
    round_lr_decay=0.70,
    weight_decay=2e-4,

    # Push direction/cosine much harder once reconstruction is reasonable.
    cosine_weight=0.30,
    near_gate_cosine_weight=0.85,
    near_gate_nmse=0.38,

    # Used only when enable_recurrent_memory=True.
    memory_enable_nmse=0.36,
    memory_enable_cosine=0.80,

    # Strict final acceptance — unchanged.
    max_local_nmse=0.20,
    min_local_cosine=0.90,
    min_teacher_agreement=0.95,
    max_mean_kl=0.05,
    max_accuracy_drop=0.02,

    gate_cases=80,
    final_cases=160,

    decision_refine_steps=30,
    decision_refine_lr_scale=0.25,
)
print(cfg)


## 3. Fit all layers and validate the cumulative student

Teacher I/O is cached per layer and released before the next candidate. Local fitting restores the best fixed-probe checkpoint, then decision gates evaluate the student with every previously accepted replacement. Earlier accepted layers stay frozen. Gate examples come from held-out train rows; final evaluation uses the official test split.

Every candidate saves progress. All-layer training takes longer than a single-layer experiment; early stopping avoids spending the full budget on layers that already pass.


In [ ]:
teacher, student, report = run_memory_fusion_v3(cfg)


## 4. Result summary
If no layer passes, the final student is restored to original Laya attention and the report explicitly says 'failed_no_accepted_layers'. Identical teacher/student metrics are therefore never presented as a successful conversion.


In [ ]:
summary = pd.DataFrame([
    {
        "model": "Laya teacher",
        **{k: report["teacher_final"].get(k) for k in [
            "accuracy", "soft_accuracy", "brier", "brier_vs_soft",
            "ece", "score_mae", "ms_per_case"
        ]},
    },
    {
        "model": report["architecture"],
        **{k: report["student_final"].get(k) for k in [
            "accuracy", "soft_accuracy", "brier", "brier_vs_soft",
            "ece", "score_mae", "ms_per_case"
        ]},
        "teacher_agreement": report["student_final"].get("teacher_agreement"),
        "teacher_KL": report["student_final"].get("mean_teacher_kl"),
    },
])
display(summary)

print("Status:", report["status"])
print("Accepted attention layers:", report["accepted_layers"])
print("All attention replaced:", report["all_attention_replaced"])
print("Remaining original layers:", report["remaining_attention_layers"])
print("Replacement parameters:", f'{report["replacement_parameters"]:,}')
print("Measured demo speedup:", round(report["latency"]["speedup"], 3), "x")
print("Gate/final disjoint:", report["gate_final_disjoint"])
print("Latency:", json.dumps(report["latency"], indent=2))


## 5. Layer-by-layer diagnostics

Fixed **PROBE** values determine local checkpoint selection. Decision-gate rows show cumulative teacher agreement, KL, accuracy and acceptance. A good local probe alone does not guarantee acceptance: earlier replacements can change downstream inputs. Rejected candidates keep their original attention.


In [ ]:
rows = []
for h in report["history"]:
    gate = h.get("gate") or {}
    local = h.get("local") or {}
    rows.append({
        "layer": h.get("layer"),
        "round": h.get("round"),
        "stage": h.get("stage"),
        "local_pass": h.get("local_pass"),
        "accepted": h.get("accepted"),
        "best_step": local.get("step"),
        "nmse": local.get("nmse"),
        "cosine": local.get("cosine"),
        "teacher_agreement": gate.get("teacher_agreement"),
        "mean_teacher_kl": gate.get("mean_teacher_kl"),
        "accuracy": gate.get("accuracy"),
        "accuracy_drop": h.get("accuracy_drop"),
    })
display(pd.DataFrame(rows))


## 6. Laya Router smoke test

The adapted Agent keeps Laya's public API. The example below attaches the already-loaded student without loading a second model.


In [ ]:
from laya import Router

state = {
    "from": "user@acme.com",
    "subject": "Duplicate charge on invoice #4411",
    "body": "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan.",
}
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "sales": "pricing, new contracts",
            "other": "everything else",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this request?",
        "criteria": ["not urgent", "soon", "critical deadline or blocking issue"],
    },
    "churn_risk": {
        "type": "noul",
        "instructions": "Does the user threaten to cancel or leave?",
    },
    "refund_requested": {
        "type": "noul",
        "instructions": "Does the user explicitly request a refund?",
    },
}

router = Router(preload=False)
router.attach("english", student)
res = router.predict(state, questions, model="english")

print("Department       :", res["answers"]["department"]["choice"])
print("Urgency score    :", res["answers"]["urgency"]["score"])
print("Churn risk       :", res["answers"]["churn_risk"]["noul"])
print("Refund requested :", res["answers"]["refund_requested"]["noul"])
print("Routing          :", res["routing"]["model"])
print("\nTeacher/student raw comparison:")
print(json.dumps(report["demo"], indent=2, ensure_ascii=False))


## 7. Saved outputs

V3 writes its adapter, report, and best checkpoint for each functional round under `/content/laya_tinycenn/memory_fusion_v3/`.


In [ ]:
from pathlib import Path
out = Path(cfg.output_dir) / "memory_fusion_v3"
print("Adapter:", out / "adapter.pt")
print("Report :", out / "report.json")
print("Round checkpoints:")
for p in sorted(out.glob("layer_*_round_*.pt")):
    print(" -", p.name)
print("\nReport preview:")
print((out / "report.json").read_text()[:5000])
